# Итоговое сравнение алгоритмов

Сравнение результатов всех алгоритмов из лабораторных работ 1-5 на одинаковых датасетах и разбиениях.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    pass
%matplotlib inline

In [ ]:
# Загрузка данных (те же разбиения, что и в лабах)
wine_cols = ['class', 'alcohol', 'malic_acid', 'ash', 'alcalinity', 'magnesium',
             'total_phenols', 'flavanoids', 'nonflavanoid_phenols', 'proanthocyanins',
             'color_intensity', 'hue', 'od280', 'proline']
wine = pd.read_csv('../data/wine.data', header=None, names=wine_cols)
parkinsons = pd.read_csv('../data/parkinsons_updrs.data')

X_wine = wine.drop('class', axis=1)
y_wine = wine['class']
X_wine_train, X_wine_test, y_wine_train, y_wine_test = train_test_split(
    X_wine, y_wine, test_size=0.3, random_state=42, stratify=y_wine)

X_park = parkinsons.drop(['subject#', 'motor_UPDRS', 'total_UPDRS'], axis=1)
y_park = parkinsons['total_UPDRS']
X_park_train, X_park_test, y_park_train, y_park_test = train_test_split(
    X_park, y_park, test_size=0.2, random_state=42)

scaler_wine = StandardScaler()
X_wine_train_s = scaler_wine.fit_transform(X_wine_train)
X_wine_test_s = scaler_wine.transform(X_wine_test)

scaler_park = StandardScaler()
X_park_train_s = scaler_park.fit_transform(X_park_train)
X_park_test_s = scaler_park.transform(X_park_test)


## Сравнение моделей sklearn (улучшенные версии)

In [ ]:
models_clf = {
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=7, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)
}

models_reg = {
    'KNN': KNeighborsRegressor(n_neighbors=5),
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(max_depth=5, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=7, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=42)
}

results_clf = []
for name, model in models_clf.items():
    model.fit(X_wine_train_s, y_wine_train)
    pred = model.predict(X_wine_test_s)
    results_clf.append({
        'Алгоритм': name,
        'Accuracy': accuracy_score(y_wine_test, pred),
        'F1-macro': f1_score(y_wine_test, pred, average='macro')
    })

results_reg = []
for name, model in models_reg.items():
    model.fit(X_park_train_s, y_park_train)
    pred = model.predict(X_park_test_s)
    results_reg.append({
        'Алгоритм': name,
        'RMSE': np.sqrt(mean_squared_error(y_park_test, pred)),
        'R2': r2_score(y_park_test, pred)
    })

df_clf = pd.DataFrame(results_clf)
df_reg = pd.DataFrame(results_reg)

print('Классификация (Wine)')
print(df_clf.round(4))
print('\nРегрессия (Parkinsons)')
print(df_reg.round(4))


In [ ]:
# Графики сравнения
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

df_clf.set_index('Алгоритм')[['Accuracy', 'F1-macro']].plot(kind='bar', ax=axes[0], rot=20)
axes[0].set_title('Классификация (Wine)')
axes[0].set_ylim(0, 1.05)
axes[0].legend(loc='lower right')

df_reg.set_index('Алгоритм')['RMSE'].plot(kind='bar', ax=axes[1], rot=20, color='steelblue', legend=False)
axes[1].set_title('RMSE, Parkinsons')
axes[1].set_ylabel('RMSE')

df_reg.set_index('Алгоритм')['R2'].plot(kind='bar', ax=axes[2], rot=20, color='darkorange', legend=False)
axes[2].set_title('R2, Parkinsons')
axes[2].set_ylim(0, 1.05)
axes[2].set_ylabel('R2')

plt.tight_layout()
plt.show()


## Выводы по всем лабораторным

По графику классификации все алгоритмы высокие и идут кучно. Чуть выше остальных случайный лес и логистическая регрессия, чуть ниже KNN. Сильного провала нет ни у кого.

На регрессии уже видно разницу. Самая маленькая ошибка и самый высокий R2 у случайного леса, следом градиентный бустинг. KNN и дерево посередине. Линейная регрессия на графике хуже всех: столбец RMSE самый длинный, R2 самый низкий.
